# Pattern ⑥ 観測性・MLOps連携

## 1. Dify × MLflow 連携案の全体像

DifyのLLMトレースをDatabricks MLflowに送る方法は複数あります。

| # | 連携案 | 仕組み | 難易度 | 推奨度 |
|---|-------|--------|--------|--------|
| A | **Databricksネイティブプロバイダー** | Dify組み込みの`databricks`プロバイダーでMLflowに直接送信 | ⭐ | **◎ 推奨** |
| B | MLflowプロバイダー | Dify組み込みの`mlflow`プロバイダーでOSS MLflowサーバーに送信 | ⭐⭐ | ○ |
| C | Langfuse経由 | Dify→Langfuse→ETL→Databricks Delta | ⭐⭐⭐ | △ |
| D | Dify API + カスタムETL | Dify APIからログ取得→Databricksで加工 | ⭐⭐⭐ | △ |

### なぜ案Aが推奨か

- Dify v1.10.1で追加された**Databricks専用プロバイダー**
- PATまたはOAuth M2Mで認証（追加インフラ不要）
- トレースがDatabricks MLflow Experimentに**直接記録**される
- 中間サーバー（Langfuse等）が不要

In [ ]:
%run ./config

## 2. 今回の連携案: Databricksネイティブプロバイダー

Dify v1.10.1以降、トレーシングプロバイダーに**Databricks**が追加されました。  
内部的にはMLflow Tracingを使い、Databricksワークスペースに直接トレースを送信します。

### 対応する全プロバイダー（10種）

| プロバイダー | 備考 |
|------------|------|
| Langfuse | OSS LLM観測ツール |
| LangSmith | LangChain公式 |
| Opik | Comet ML |
| Arize / Phoenix | ML観測 |
| Weave | Weights & Biases |
| MLflow | OSS MLflow サーバー向け |
| **Databricks** | **Databricks管理MLflow向け（推奨）** |
| Aliyun / Tencent | 中国クラウド向け |

## 3. アーキテクチャ

```
Dify App（Chatbot / Agent / Workflow）
  │
  │  Databricksプロバイダー（組み込み）
  │  認証: PAT or OAuth M2M
  │
  ▼
Databricks MLflow Experiment
  │
  ├── Traces: 全LLM呼び出し・ツール実行・検索を記録
  ├── AI Judge: 品質自動評価（Safety, 日本語品質, 事実性）
  └── Dashboard: レイテンシ・トークン・品質スコアを可視化
```

### トレース対象

| アプリタイプ | 記録される内容 |
|------------|-------------|
| Chatbot | LLM呼び出し、ナレッジ検索 |
| Agent | LLM呼び出し、ツール呼び出し（MCP含む）、推論ステップ |
| Workflow | 全ノード実行（HTTP, Code, LLM, 条件分岐等） |

## 4. 連携プロセス

### 4.1 Databricks側: MLflow Experimentの作成

1. Databricksワークスペース左メニュー **「Experiments」** をクリック
2. 右上の **「Create Experiment」** をクリック
3. **Name**: `/Users/<your_email>/dify_tracing` と入力
4. **Create** をクリック
5. 作成後、画面上部のExperiment IDを控える

In [ ]:
%pip install --upgrade mlflow>=3.0 databricks-agents --quiet
dbutils.library.restartPython()

In [ ]:
import mlflow
print(f"MLflow version: {mlflow.__version__}")

# 今回作成済みのExperimentを使用
experiment_name = "/Users/yunyi.yao@databricks.com/dify_workflow_tracing"
experiment_id = "440796949368760"

host = spark.conf.get("spark.databricks.workspaceUrl")

print(f"✅ Experiment: {experiment_name} (ID: {experiment_id})")
print(f"\n{'=' * 60}")
print(f"Dify 監視設定（Databricksプロバイダー）")
print(f"{'=' * 60}")
print(f"\n【実験 ID】{experiment_id}")
print(f"【DatabricksワークスペースのURL】https://{host}")
print(f"【パーソナルアクセストークン】Dify専用の長期PATを使用")

### 4.2 Dify側: 監視設定（UI手順）

**アプリごとに設定が必要です。**

1. 対象のアプリを開く
2. 左メニュー **「監視」** をクリック
3. **「設定」** をクリック → プロバイダー一覧が表示される
4. **「Databricks」** を選択
5. 入力:
   - **実験 ID**: 上のセルで出力されたID
   - **DatabricksワークスペースのURL**: 上のセルで出力されたURL
   - **OAuthクライアントID / シークレット**: 空欄でOK
   - **パーソナルアクセストークン（レガシー）**: Dify専用PAT
6. **「保存 & 有効に」** をクリック

> **ポイント**: 本番運用ではPATではなく**OAuth M2M**（SP + client_id/secret）を推奨。  
> PATは「レガシー」と表記されている通り、Databricks側でもOAuth推奨に移行中。

## 5. MLflowによる品質評価と監視

Difyからトレースが記録されたら、MLflowのAI Judge機能で**自動品質評価**ができます。

### 重要: Experiment IDはワークスペース共有

DifyのDatabricksプロバイダー設定（host, PAT, Experiment ID）は**Difyワークスペース全体で1つ**です。  
アプリごとに異なるExperimentに送ることはできません。

```
Difyワークスペース
  ├── ③ MCP Agent        ─┐
  ├── ② UC関数実行        ─┤──▶ 1つのMLflow Experiment（共有）
  ├── ① LLM endpoint利用  ─┘
```

### アプリの識別方法

現時点（Dify v1.13.0）では、**トレースにアプリ名が含まれない**。  
代わりに `tags['mlflow.traceName']`（スパン名）でトレースの種類を識別する:

| スパン名 | 内容 |
|---------|------|
| `message` | Agent/Chatbot のLLM呼び出し（最終回答） |
| `mcp_sse_call_tool` | MCPツール呼び出し（UC Functions, Vector Search） |
| `mcp_sse_list_tools` | MCPツール一覧取得 |
| `generate_conversation_name` | 会話タイトル自動生成（評価対象外） |
| `workflow` | Workflowアプリのルートスパン |

In [ ]:
import mlflow
import json

traces = mlflow.search_traces(locations=[experiment_id], max_results=50)
print(f"✅ 記録済みトレース: {len(traces)}件\n")

# トレース一覧表示（スパン名で識別）
for i, (_, trace) in enumerate(traces.iterrows()):
    trace_name = trace.get("tags", {}).get("mlflow.traceName", "?")
    duration = trace.get("execution_duration", 0)
    req = str(trace.get('request', ''))[:120]
    print(f"  [{i+1}] Span: {trace_name} | {duration}ms")
    print(f"       Request: {req}...")
    print()

In [ ]:
import json

# === 全トレースをスパン名で分類 ===
llm_traces = []       # LLM呼び出し（最終回答）
tool_traces = []      # ツール呼び出し（MCP等）
search_traces = []    # Vector Search結果

for _, trace in traces.iterrows():
    request = trace.get("request")
    response = trace.get("response", "")
    trace_name = trace.get("tags", {}).get("mlflow.traceName", "")
    trace_id = trace["trace_id"]
    duration = trace.get("execution_duration", 0)
    
    # 会話タイトル生成は除外
    if trace_name == "generate_conversation_name":
        continue
    
    # 1. LLM呼び出し: スパン名が "message" または requestがmessages配列
    if trace_name == "message" or (isinstance(request, list) and len(request) > 0 and isinstance(request[0], dict) and "role" in request[0]):
        if isinstance(request, list):
            user_messages = [m["content"] for m in request if m.get("role") == "user"]
        else:
            user_messages = [str(request)[:500]]
        if user_messages and response:
            llm_traces.append({
                "trace_id": trace_id, "trace_name": trace_name,
                "request": user_messages[-1][:500],
                "response": str(response)[:2000],
                "duration_ms": duration, "type": "llm"
            })
    
    # 2. ツール呼び出し
    elif isinstance(request, dict) and "tool_name" in request:
        tool_name = request.get("tool_name", "")
        tool_args = request.get("arguments", "")
        if "vs_index" in tool_name or "knowledge_base" in tool_name:
            query = ""
            try:
                query = json.loads(tool_args).get("query", "") if isinstance(tool_args, str) else ""
            except: pass
            search_traces.append({
                "trace_id": trace_id, "trace_name": trace_name,
                "tool_name": tool_name, "query": query,
                "results": str(response)[:2000],
                "duration_ms": duration, "type": "search"
            })
        else:
            tool_traces.append({
                "trace_id": trace_id, "trace_name": trace_name,
                "tool_name": tool_name,
                "arguments": str(tool_args)[:300],
                "duration_ms": duration, "type": "tool"
            })
    
    # 3. Workflowトレース
    elif trace_name == "workflow":
        llm_traces.append({
            "trace_id": trace_id, "trace_name": trace_name,
            "request": str(request)[:500],
            "response": str(response)[:2000],
            "duration_ms": duration, "type": "llm"
        })

print(f"✅ トレース分類結果:")
print(f"  LLM/Workflow:      {len(llm_traces)}件")
print(f"  Vector Search:     {len(search_traces)}件")
print(f"  その他ツール呼び出し: {len(tool_traces)}件")

for t in llm_traces:
    print(f"\n  [LLM] Span: {t['trace_name']}")
    print(f"    Q: {t['request'][:100]}")
    print(f"    A: {t['response'][:100]}")

for s in search_traces:
    print(f"\n  [Search] Query: {s['query'][:80]}")

In [ ]:
from mlflow.genai.scorers import Guidelines, Safety
import pandas as pd

# === 評価A: 最終回答の品質評価 ===
print("=" * 60)
print("評価A: 最終回答の品質（Safety, 日本語品質, 事実性）")
print("=" * 60)

if llm_traces:
    # MLflow 3のevaluate()は inputs / outputs カラムを期待
    eval_data_llm = pd.DataFrame([{
        "inputs": {"request": t["request"]},
        "outputs": {"response": t["response"]}
    } for t in llm_traces])

    scorers_llm = [
        Safety(),
        Guidelines(
            name="japanese_quality",
            guidelines="回答は日本語で、丁寧な敬語で書かれていること。技術用語は適切に使用されていること。"
        ),
        Guidelines(
            name="no_hallucination",
            guidelines="回答は提供されたコンテキスト情報に基づいており、事実の捏造や推測が含まれていないこと。"
        ),
    ]

    mlflow.set_experiment(experiment_name)
    with mlflow.start_run(run_name="eval_A_response_quality"):
        results_llm = mlflow.genai.evaluate(
            data=eval_data_llm,
            scorers=scorers_llm
        )
    print("✅ 評価A完了")
    print(results_llm.tables["eval_results"].to_string())
else:
    print("⚠️ LLMトレースが0件のため評価スキップ")

In [ ]:
# === 評価B: RAG検索品質（検索結果と最終回答の整合性）===
print("=" * 60)
print("評価B: RAG検索品質（検索結果が回答に正しく反映されているか）")
print("=" * 60)

if search_traces and llm_traces:
    # 検索結果をコンテキストとしてLLM回答と結合
    rag_eval_data = []
    for llm in llm_traces:
        context_texts = []
        for s in search_traces:
            try:
                results = json.loads(s['results']) if isinstance(s['results'], str) else s['results']
                if isinstance(results, list):
                    for r in results:
                        if isinstance(r, dict) and 'content' in r:
                            context_texts.append(r['content'][:300])
            except Exception:
                context_texts.append(s['results'][:300])
        
        if context_texts:
            context_str = "\n---\n".join(context_texts)
            rag_eval_data.append({
                "inputs": {"request": llm['request'], "context": context_str},
                "outputs": {"response": llm['response']}
            })

    if rag_eval_data:
        eval_data_rag = pd.DataFrame(rag_eval_data)
        
        # RetrievalGroundednessはRETRIEVERスパンが必要なため、Guidelinesで代替
        rag_scorer = Guidelines(
            name="grounded_in_context",
            guidelines="回答は提供された検索コンテキスト（inputs.context）の情報に基づいていること。"
                       "コンテキストに含まれない情報を捏造していないこと。"
                       "コンテキストの内容を正確に要約・引用していること。"
        )
        
        with mlflow.start_run(run_name="eval_B_rag_groundedness"):
            results_rag = mlflow.genai.evaluate(
                data=eval_data_rag,
                scorers=[rag_scorer]
            )
        print("✅ 評価B完了")
        print(results_rag.tables["eval_results"].to_string())
    else:
        print("⚠️ 検索結果とLLM回答の紐付けが0件")
else:
    print("⚠️ 検索トレースまたはLLMトレースが0件のため評価スキップ")
    print(f"  Search: {len(search_traces)}件, LLM: {len(llm_traces)}件")

In [ ]:
# === 評価C: ツール呼び出しのパフォーマンス分析 ===
print("=" * 60)
print("評価C: ツール呼び出しパフォーマンス")
print("=" * 60)

all_spans = llm_traces + search_traces + tool_traces
if all_spans:
    perf_df = pd.DataFrame(all_spans)
    print("\nスパン種別ごとのレイテンシ:")
    summary = perf_df.groupby('type').agg(
        count=('duration_ms', 'count'),
        avg_ms=('duration_ms', 'mean'),
        max_ms=('duration_ms', 'max')
    ).round(0)
    print(summary.to_string())
    
    # ボトルネック検出
    slowest = perf_df.loc[perf_df['duration_ms'].idxmax()]
    print(f"\n⚠️ 最も遅いスパン: {slowest.get('type')} ({slowest['duration_ms']}ms)")
else:
    print("⚠️ スパンデータなし")

### 評価のまとめ

| 評価 | 対象 | Scorer | 評価内容 |
|------|------|--------|--------|
| **A** | LLM最終回答 | Safety, Guidelines | 安全性、日本語品質、事実性 |
| **B** | 検索結果 + 回答 | Guidelines（カスタム） | 回答が検索コンテキストに基づいているか |
| **C** | 全スパン | パフォーマンス分析 | レイテンシ、ボトルネック検出 |

> **注意**: MLflow 3の`RetrievalGroundedness`スコアラーは`RETRIEVER`タイプのスパンが必要ですが、  
> Difyのトレースにはこのスパンタイプが含まれないため、`Guidelines`ベースのカスタム評価で代替しています。

### 監視ダッシュボード（発展）

トレースと評価結果をDeltaテーブルに格納すれば、SQLダッシュボードで監視できます:

```sql
-- アプリ別の平均レイテンシとトークン使用量
SELECT
  date_trunc('hour', timestamp) as hour,
  COUNT(*) as requests,
  AVG(execution_time_ms) as avg_latency_ms,
  AVG(safety_score) as avg_safety,
  AVG(japanese_quality_score) as avg_quality
FROM dify_evaluation_results
GROUP BY 1
ORDER BY 1 DESC
```

自動化する場合はDatabricks Workflowで定期実行:
```
Workflow（毎日9:00 JST）
  ├── Step 1: MLflowから新規トレース取得
  ├── Step 2: AI Judgeで品質評価
  ├── Step 3: 結果をDeltaテーブルに保存
  └── Step 4: 閾値以下のスコアがあればアラート通知
```

## 6. この連携案のメリット

| 観点 | 説明 |
|------|------|
| **ゼロインフラ** | Langfuse等の中間サーバー不要。Dify→Databricks直接 |
| **ネイティブ統合** | Dify組み込みプロバイダー。UIで設定するだけ |
| **全アプリ対応** | Chatbot / Agent / Workflow 全てのアプリタイプでトレース可能 |
| **AI Judge** | MLflowのScorerで品質自動評価（Safety, 日本語品質, 事実性） |
| **一元管理** | 複数Difyアプリのトレースを1つのMLflow Experimentに集約 |
| **ガバナンス** | UC権限でトレースデータのアクセスも制御可能 |

### Ricohへのメッセージ

> Difyでアプリが乱立しても、**観測性はDatabricks MLflowで一元管理**できる。  
> 全アプリのLLM呼び出し・ツール実行・検索をトレースし、  
> AI Judgeで品質を自動評価することで、**ガバナンスと品質の両立**が可能。  
> これはDify単体では実現できない、Databricks連携の大きな価値。

---

## 現時点の制約と注意事項

Dify × Databricks MLflow連携は**v1.10.1で追加されたばかりの機能**であり、以下の不安定な部分・未検証事項があります。

### 確認済みの制約

| 制約 | 詳細 |
|------|------|
| **Experiment IDがワークスペース共有** | アプリごとに異なるExperimentに送信できない |
| **トレースにアプリ名が含まれない** | Agent/Chatbotのトレースでは`app_name`が未設定。スパン名での識別が必要 |
| **RETRIEVERスパンはMCP経由では生成されない** | MCP経由のVector SearchはTOOLタイプになり、`RetrievalGroundedness`スコアラーが使えない |
| **External Knowledge + トレーシングの併用不可** | Dify v1.13.0のバグ（`dataset_retrieval_trace`でNullPointer） |

### 未検証事項

| 項目 | 状況 |
|------|------|
| Workflowアプリの詳細トレース | Workflowの子ノード（LLM, ナレッジ検索等）が個別スパンとして記録されるか未検証 |
| `RetrievalGroundedness`の動作 | Dify標準ナレッジ検索（External Knowledgeでない）で`RETRIEVER`スパンが正しく生成されるか未検証 |
| 大量トレース時のパフォーマンス | 数千件のトレースがMLflow Experimentに蓄積された場合の検索・評価パフォーマンス |
| OAuth M2M認証 | PATでは検証済みだが、OAuth（client_id/secret）での接続は未検証 |
| Difyバージョンアップ時の互換性 | v1.13.0以降のアップデートでトレース構造やバグ修正の影響 |

> **まとめ**: Databricksプロバイダーによるトレース送信とAI Judge評価の基本フローは動作確認済み。  
> ただし、Dify側のMLflow連携はまだ成熟しておらず、本番運用にはDifyのバージョンアップを追跡しながら段階的に導入することを推奨。